# LLMBackend Combined Hypothesis Graph Queries

This notebook resolves one pickup instruction, projects **both** FrameNet and Flanagan sidecars into the same `HypothesisGraph`, then queries the result through the view API.

- `FlanaganGraphView(graph=graph)` exposes typed Flanagan entity queries
- `FrameNetGraphView(graph=graph)` exposes typed FrameNet entity queries
- one shared `BuildOrchestrator` inserts both projections into the same graph
- `graph.claims_for_action(action)` scopes queries to a specific resolved action

In [1]:
from uniworld import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body

world, robot_view, context = load_pr2_apartment_world()
context.evaluate_conditions = False

symbol_type = Body
# print("World loaded:", type(world).__name__)
# print("Robot:", robot_view)


`polytope` failed to import `cvxopt.glpk`.
will use `scipy.optimize.linprog`
Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


World loaded: World
Robot: PR2(neck=Neck(name=PrefixedName('pr2/neck'), id=UUID('f4bcd944-ec03-471f-9f11-806f496530f0'), root=Body(name=PrefixedName('pr2/head_pan_link'), id=UUID('1465e918-4ca3-40ee-bad8-ac4bc31a20d8'), index=21), _robot=..., joint_states=[], tip=Body(name=PrefixedName('pr2/head_tilt_link'), id=UUID('47c58ea9-a471-49e6-9c10-aac6961870be'), index=22), manipulator=None, sensors=[Camera(name=PrefixedName('pr2/wide_stereo_optical_frame'), id=UUID('da96eae8-4cf7-47a8-8fa6-d9344fde8807'), root=Body(name=PrefixedName('pr2/wide_stereo_optical_frame'), id=UUID('ad3f1cdb-600c-4895-ba83-7262eeae9867'), index=29), _robot=..., joint_states=[], forward_facing_axis=Vector3(@1=0, [@1, @1, 1, @1]), field_of_view=FieldOfView(vertical_angle=0.75049, horizontal_angle=0.99483), minimal_height=1.27, maximal_height=1.6, default_camera=False)], pitch_body=Body(name=PrefixedName('pr2/head_tilt_link'), id=UUID('47c58ea9-a471-49e6-9c10-aac6961870be'), index=22), yaw_body=Body(name=PrefixedName('

In [2]:
from dotenv import load_dotenv
from llmr.reasoning.llm_provider import make_llm, LLMProvider

load_dotenv("../.env")

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini", temperature=0.0)
print("LLM ready:", getattr(llm, "model_name", llm))


LLM ready: gpt-4o-mini


## Combined hypothesis-graph setup

The next cell creates a shared `BuildOrchestrator` that wires both the FrameNet and Flanagan builders automatically.

After evaluation we will query that same graph through two typed views:

- `FrameNetGraphView(graph=graph)`
- `FlanaganGraphView(graph=graph)`

In [ ]:
from krrood.entity_query_language.query.match import Match
from llmr.backend import LLMBackend
from llmr.hypotheses import (
    FrameClaim,
    RoleClaim,
    GroundingState,
    PhaseClaim,
    PlanClaim,
    BuildOrchestrator,
    FlanaganGraphView,
    FrameNetGraphView,
)
from llmr.hypotheses.adapters import to_dot
from llmr.reasoning.flanagan_reasoner import FlanaganReasoner
from llmr.reasoning.framenet_reasoner import FrameNetReasoner
from pycram.datastructures.grasp import GraspDescription
from pycram.robot_plans.actions.core.pick_up import PickUpAction

orchestrator = BuildOrchestrator.with_default_builders()
graph = orchestrator.graph

framenet = FrameNetGraphView(graph=graph)
flanagan = FlanaganGraphView(graph=graph)


def fresh_pickup_match():
    return Match(PickUpAction)(
        object_designator=...,
        arm=...,
        grasp_description=Match(GraspDescription)(
            approach_direction=...,
            vertical_alignment=...,
            manipulator=...,
        ),
    )


def run_instruction(instruction: str):
    backend = LLMBackend(
        llm=llm,
        symbol_type=symbol_type,
        instruction=instruction,
        strict_required=True,
        reasoners=[
            FrameNetReasoner(llm=llm),
            FlanaganReasoner(llm=llm),
        ],
        sg_model_orchestrator=orchestrator,
    )
    action = next(iter(backend.evaluate(fresh_pickup_match())))
    return action, backend


def role_rows(roles):
    return [
        {
            "display_id": role.display_id,
            "role": role.role_name,
            "family": role.role_family,
            "text": role.filler_text,
            "kind": role.filler_kind,
            "status": role.meta.status.value,
            "grounding": role.meta.grounding.value,
            "run_id": role.meta.short_run_id,
        }
        for role in roles
    ]


def phase_rows(phases):
    return [
        {
            "display_id": phase.display_id,
            "index": phase.phase_index,
            "phase": phase.phase_name,
            "target": phase.target_object,
            "contact": phase.contact,
            "motion_type": phase.motion_type,
            "urgency": phase.urgency,
            "run_id": phase.meta.short_run_id,
        }
        for phase in phases
    ]


print("Combined graph and helpers ready.")

In [5]:
INSTRUCTION = "pick up the milk from the table"

In [ ]:
action, backend = run_instruction(INSTRUCTION)

current_frame = framenet.frames()[0]
current_plan = flanagan.plans()[0]

print("Resolved action:", type(action).__name__)
print("Resolved object:", action.object_designator)
print("Resolved arm:", action.arm)
print("FrameNet run:", current_frame.meta.short_run_id)
print("Flanagan run:", current_plan.meta.short_run_id)
print("Graph entity count:", len(graph))

In [ ]:
framenet.frames()

In [7]:
import json

print("FrameNet sidecar")
print(json.dumps(backend.semantics.frames.model_dump(by_alias=True), indent=2, default=str))

print("\nFlanagan sidecar")
print(json.dumps(backend.semantics.motion_phases.model_dump(), indent=2, default=str))


FrameNet sidecar
{
  "framenet": "picking_up_object",
  "frame": "Getting",
  "lexical-unit": "pick_up.v",
  "core": {
    "agent": "robot",
    "theme": "milk.stl",
    "patient": null,
    "instrument": "right_gripper",
    "source": "table",
    "goal": "robot's grasp",
    "result": "robot holds milk.stl",
    "other_core_elements": null
  },
  "peripheral": {
    "location": "kitchen area",
    "manner": "carefully",
    "direction": "upward",
    "time": null,
    "purpose": "for transport",
    "quantity": "one milk",
    "portion": null,
    "speed": null,
    "path": null,
    "other_peripheral_elements": null
  }
}

Flanagan sidecar
{
  "instruction": "pick up the milk from the table",
  "phases": [
    {
      "phase": "Approach",
      "target_object": "milk",
      "description": "move to milk on table",
      "symbol": "->[ robot approachs milk]",
      "goal_state": {
        "arm_near_milk": true,
        "distance_to_milk_m": 0.2
      },
      "preconditions": {
     

In [32]:
backend.semantics.model_dump(by_alias=True)

{'action_type': 'PickUpAction',
 'instruction': 'pick up the milk from the table',
 'classification': None,
 'slot_filling': {'action_type': 'PickUpAction',
  'slots': [{'field_name': 'object_designator',
    'value': None,
    'entity_description': {'name': 'milk.stl',
     'semantic_type': 'Milk',
     'spatial_context': 'on the table',
     'attributes': None},
    'reasoning': "The instruction specifies to pick up 'the milk', which corresponds to the object 'milk.stl' in the scene."},
   {'field_name': 'arm',
    'value': 'RIGHT',
    'entity_description': None,
    'reasoning': 'Using the right arm is a common choice for picking up objects, especially if the robot is designed for right-handed operation.'},
   {'field_name': 'grasp_description.approach_direction',
    'value': 'FRONT',
    'entity_description': None,
    'reasoning': 'Approaching from the front is typical when picking up an object from a table, allowing for a clear line of sight and access.'},
   {'field_name': 'gr

## Graph-native inspection

Before using EQL, it is useful to inspect the action-local subgraph through both family views. This shows that the same `HypothesisGraph` now contains claims from both reasoner families.


In [ ]:
{
    "reasoner_runs": [run.reasoner_name for run in graph.reasoner_runs],
    "framenet_claim_count": len(framenet.claims()),
    "flanagan_claim_count": len(flanagan.claims()),
    "frame_ids": [node.display_id for node in framenet.frames()],
    "plan_ids": [node.display_id for node in flanagan.plans()],
}

In [ ]:
role_rows(framenet.roles())

In [ ]:
phase_rows(flanagan.phases())

## Query setup

These EQL variables come from the action-local family views. The graph core stays generic; the views provide the typed domains.


In [ ]:
from krrood.entity_query_language.factories import an, entity, variable

frame = variable(FrameClaim, domain=graph)
role  = variable(RoleClaim,  domain=graph)
plan  = variable(PlanClaim,  domain=graph)
phase = variable(PhaseClaim, domain=graph)

{
    "frames": [item.display_id for item in framenet.frames()],
    "roles":  [item.display_id for item in framenet.roles()],
    "plans":  [item.display_id for item in flanagan.plans()],
    "phases": [item.display_id for item in flanagan.phases()],
}

In [26]:
# Query 1: inferred FrameNet frame for this instruction
frame_query = an(
    entity(frame).where(
        frame.instruction_text == INSTRUCTION,
    )
)
list(frame_query.evaluate())


[FrameClaimNode(id='framenet_reasoner:58bc14f3f09142c9b1c5bd978b67a3c4:node:frame', meta=HypothesisMeta(source_reasoner='framenet_reasoner', status=<ClaimStatus.HYPOTHESIS: 'hypothesis'>, grounding=<GroundingState.TEXT_ONLY: 'text_only'>, confidence=None, run_id='58bc14f3f09142c9b1c5bd978b67a3c4', prompt_version='framenet_v1', model_name='gpt-4o-mini', created_at=datetime.datetime(2026, 4, 28, 8, 9, 15, 206358, tzinfo=datetime.timezone.utc)), frame='Getting', lexical_unit='pick_up.v', framenet_label='picking_up_object', action_type='PickUpAction', instruction_text='pick up the milk from the table')]

In [27]:
# Query 2: grounded FrameNet roles
grounded_role_query = an(
    entity(role).where(
        role.meta.grounding == GroundingState.SYMBOL_GROUNDED,
    )
)
role_rows(list(grounded_role_query.evaluate()))


[{'display_id': 'framenet_reasoner:58bc14f3:node:role:core:theme',
  'role': 'theme',
  'family': 'core',
  'text': 'milk.stl',
  'kind': 'entity',
  'status': 'supported',
  'grounding': 'symbol_grounded',
  'run_id': '58bc14f3'}]

In [28]:
# Query 3: all Flanagan motion phases
phase_query = an(entity(phase))
phase_rows(list(phase_query.evaluate()))


[{'display_id': 'flanagan_reasoner:dc89764d:node:phase:0:Approach',
  'index': 0,
  'phase': 'Approach',
  'target': 'milk',
  'contact': False,
  'motion_type': None,
  'urgency': 'medium',
  'run_id': 'dc89764d'},
 {'display_id': 'flanagan_reasoner:dc89764d:node:phase:1:Grasp',
  'index': 1,
  'phase': 'Grasp',
  'target': 'milk',
  'contact': True,
  'motion_type': 'pinch',
  'urgency': 'high',
  'run_id': 'dc89764d'},
 {'display_id': 'flanagan_reasoner:dc89764d:node:phase:2:Lift',
  'index': 2,
  'phase': 'Lift',
  'target': 'milk',
  'contact': True,
  'motion_type': 'upward_lift',
  'urgency': 'high',
  'run_id': 'dc89764d'}]

In [29]:
# Query 4: contact phases only
contact_phase_query = an(
    entity(phase).where(
        phase.contact == True,
    )
)
phase_rows(list(contact_phase_query.evaluate()))


[{'display_id': 'flanagan_reasoner:dc89764d:node:phase:1:Grasp',
  'index': 1,
  'phase': 'Grasp',
  'target': 'milk',
  'contact': True,
  'motion_type': 'pinch',
  'urgency': 'high',
  'run_id': 'dc89764d'},
 {'display_id': 'flanagan_reasoner:dc89764d:node:phase:2:Lift',
  'index': 2,
  'phase': 'Lift',
  'target': 'milk',
  'contact': True,
  'motion_type': 'upward_lift',
  'urgency': 'high',
  'run_id': 'dc89764d'}]

In [ ]:
# Query 5: compare FrameNet theme fillers with Flanagan phase targets
theme_query = an(
    entity(role).where(
        role.role_name == "theme",
    )
)
theme_roles = list(theme_query.evaluate())
phase_targets = sorted({item.target_object for item in flanagan.phases()})

{
    "framenet_theme_texts": [item.filler_text for item in theme_roles],
    "flanagan_phase_targets": phase_targets,
    "shared_targets": sorted(set(item.filler_text for item in theme_roles) & set(phase_targets)),
}

## Visualization

The generic graph still supports DOT export. Rendering the action-local subgraph keeps the image readable while preserving both families in the same picture.


In [ ]:
from graphviz import Source
from IPython.display import display

try:
    display(Source(to_dot(graph), format="svg"))
except Exception:
    print(to_dot(graph))